# **Recipe generator using LLM based prompt engineering**

This exercise demonstrated how to build an AI-powered recipe generator in Google Colab using the google/flan-t5-large model and a Gradio web interface.
The core focus was on Prompt Engineering—the art of structuring instructions to guide a smaller Language Model's behavior.

Through iterative testing, you learned how to overcome the model's natural tendency to be brief or repetitive by using advanced techniques like Few-Shot Prompting (giving concrete examples), Constraint Prompting (forcing specific formats), and tuning generation parameters (like length_penalty and num_beams).

Ultimately, the exercise proved a fundamental rule of AI development: when working with smaller models, how you ask the question and structure the prompt is just as critical as the model's raw intelligence.

In [ ]:
# Install required libraries (run once)
!pip install -q transformers gradio accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading google/flan-t5-large... (This may take a minute)")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load tokenizer and model directly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-large",
    dtype=dtype
).to(device)

print(f"Model loaded successfully on {device.upper()}!")

Loading google/flan-t5-large... (This may take a minute)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully on CUDA!


In [ ]:
def generate_recipe(user_input, meal_type, dietary_restrictions):
    user_input = user_input.strip()
    if not user_input:
        return "Please enter a dish name or ingredients!"

    # 1. T5 NATIVE PROMPT: Short, direct, and uses the "question/answer" format.
    # We explicitly end the prompt with "Ingredients:\n" to force it to start listing immediately.
    prompt = f"question: How do I make {user_input}? Give ingredients and step-by-step instructions. answer: Ingredients:\n"

    inputs = tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,       # CRITICAL: Turns off random sampling. Forces logical, structured text.
            num_beams=4,           # CRITICAL: Beam search helps the model plan the whole sentence before writing it.
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    recipe_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean up the output so it doesn't repeat the prompt
    if "Ingredients:" in recipe_text:
        recipe_text = recipe_text.split("Ingredients:", 1)[1].strip()
        recipe_text = "Ingredients:\n" + recipe_text

    return recipe_text

**Some of the recipes/ingredients working:**
*   Chicken masala
*   chicken
*   paneer
*   bun, cheese, chicken
*   chicken biriyani
*   bread, egg




In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=generate_recipe,
    inputs=[
        # Keep the placeholder short so the user knows not to type full sentences
        gr.Textbox(label="🍲 Dish Name OR Ingredients", placeholder="e.g., 'Simple Pasta' OR 'chicken, rice, peppers'", lines=1),
        gr.Dropdown(choices=["Any", "Breakfast", "Lunch", "Dinner", "Snack", "Dessert"], label="🍽️ Meal Type", value="Any"),
        gr.Textbox(label="🚫 Dietary Restrictions (optional)", placeholder="e.g., Vegan, Gluten-free")
    ],
    outputs=gr.Textbox(label="👨‍🍳 Your Detailed Recipe", lines=25),
    title="🍳 AI Recipe Generator",
    description="Type a **short dish name** (e.g., 'Garlic Pasta') or your **ingredients**. The AI will generate a structured, step-by-step recipe!"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b1b68e51b64319c489.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q transformers gradio sentencepiece

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def generate_recipe(user_input):

    if not user_input.strip():
        return "Please enter a recipe name or ingredients!"

    prompt = f"""
    Create a recipe for: {user_input}

    Write:
    Ingredients:
    1. ...
    2. ...

    Instructions:
    1. ...
    2. ...
    3. ...
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    output = model.generate(
        **inputs,
        max_new_tokens=250,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.5,
        early_stopping=True
    )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

In [ ]:
demo = gr.Interface(
    fn=generate_recipe,
    inputs=gr.Textbox(),
    outputs=gr.Textbox()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1615b2deef6de6c1d0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
